# Earth Forge research quickstart

Install the local platform wheel first. Python orchestrates the shared Rust engine. These are synthetic examples; field accuracy and SOTA superiority are unestablished. Arrays use metres, z positive down, and nodal (z, y, x) ordering.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from earthforge import Engine, Grid, EnsemblePlan, run_ensemble, load_dataset, condition_histories, History

engine = Engine()
print(engine.capabilities)
run_directory = TemporaryDirectory(prefix="earthforge-notebook-")
out = Path(run_directory.name)
print("Temporary outputs:", out)


## Generate and export a 10 m crop

This is a 160 × 160 × 1,000 m crop, not a full field-scale 10 m volume. Material subdivision stays 10 m. Change it only with a convergence study.

In [ ]:
grid = Grid(17, 17, 101, 10, 10, 10, (4900, 4900, 400))
history = engine.build_history("rift-v1", seed=1001, steps=6, faults=1)
engine.export_dataset(history, grid, out / "volume")
volume = load_dataset(out / "volume")
volume.save_npz(out / "model.npz")
volume.save_vti(out / "model.vti")
print(volume.fields["velocity"].shape, volume.content_digest())


## Freeze a finite prior ensemble

Parameter draws and independent hashed recipe seeds are recorded. Failed jobs remain explicit. Repeating with `resume=True` verifies completed data and resumes unfinished members.

In [ ]:
plan = EnsemblePlan(count=4, seed=1001, options={"steps":6,"faults":1}, priors={"slip_scale":[0.7,1.4],"sediment_supply_scale":[0.8,1.2]})
study = run_ensemble(engine, plan, grid, out / "ensemble")
assert study["failures"] == 0
print(study["weighting"])


## Synthetic conditioning demonstration

The first candidate supplies the synthetic observation and remains in the library. This is a workflow sanity check, not independent validation. Gaussian error assumptions and finite-library effective sample size do not establish calibrated credible intervals.

In [ ]:
members = [out / "ensemble" / c["id"] / "history.json" for c in study["identity"]["plan"]["candidates"]]
histories = [History.load(p) for p in members]
position = (5000,5000,1000)
truth = engine.points(histories[0], [position])[0]["velocity"]
observations = [{"kind":"velocity","x":5000,"y":5000,"z":1000,"value":truth,"sigma":100}]
weighted = condition_histories(engine, histories, observations)
print("Weights:", weighted["weights"])
print("Effective sample size:", weighted["effectiveSampleSize"])
